In [1]:
import pandas as pd
import h5py
import numpy as np
import os

In [ ]:
import h5py
import numpy as np
import pandas as pd


def label_time(frame_idx, starts, stops, phase_labels):
    """Return the phase label for a frame index (half-open [start, stop))."""
    for s, e, label in zip(starts, stops, phase_labels):
        if s <= frame_idx < e:      # half-open interval avoids overlap
            return label
    return "nonphase"


def extract_labels(pose_file, time_df, key_obs_map, pose_type):
    """Extract frame-level phase labels from annotation timestamps."""

    # Read frame count + fps WHILE THE FILE IS OPEN. Previously `data` was a
    # lazy dataset handle read after the `with` block closed, so `data.shape`
    # raised "Unable to synchronously get dataspace (identifier is not of
    # specified type)". Materialise n_frames here instead.
    with h5py.File(pose_file, "r") as f:
        if pose_type not in f:
            print(f"Missing dataset '{pose_type}' in {pose_file}")
            return None
        n_frames = f[pose_type].shape[0]
        fps = f.attrs.get("fps")

    if fps is None or pd.isna(fps):
        print(f"Missing fps in {pose_file}")
        return None

    filename = pose_file.split("/")[-1]
    timestamp_obs = key_obs_map.get(filename)

    # Missing (None/NaN) or blank observation -> nothing to label from.
    if (timestamp_obs is None
            or (isinstance(timestamp_obs, float) and pd.isna(timestamp_obs))
            or (isinstance(timestamp_obs, str) and timestamp_obs.strip() == "")):
        print(f"No timestamp observation found for {filename}")
        return None

    phase_rows = (
        time_df[time_df["Observation"] == timestamp_obs]
        .sort_values("Time_Relative_sf")
        .reset_index(drop=True)
    )

    # Frame index of every annotated event (chronological). Used to bound a
    # phase whose _stop is missing so it can't swallow later phases.
    event_frames = [int(round(t * fps)) for t in phase_rows["Time_Relative_sf"]]

    starts, stops, phase_labels = [], [], []

    for i in range(len(phase_rows)):
        behavior = str(phase_rows.loc[i, "Behavior"])

        if not behavior.endswith("_start"):
            continue

        phase = behavior[:-len("_start")]
        start_frame = int(round(phase_rows.loc[i, "Time_Relative_sf"] * fps))

        stop_rows = phase_rows[
            (phase_rows.index > i)
            & (phase_rows["Behavior"].astype(str) == f"{phase}_stop")
        ]

        if not stop_rows.empty:
            stop_frame = int(round(stop_rows.iloc[0]["Time_Relative_sf"] * fps))
        else:
            # No _stop annotation: run until the next annotated event if one
            # exists, otherwise to the end of the video (as requested).
            later = [ef for ef in event_frames if ef > start_frame]
            stop_frame = min(later) if later else n_frames
            where = "next event" if later else "end of video"
            print(f"  {phase} in {filename}: missing _stop -> "
                  f"stop set to frame {stop_frame} ({where})")

        # Clamp to a valid, non-empty range within the clip.
        start_frame = max(0, min(start_frame, n_frames))
        stop_frame = max(start_frame, min(stop_frame, n_frames))

        starts.append(start_frame)
        stops.append(stop_frame)
        phase_labels.append(phase)

    labels = [
        label_time(frame, starts, stops, phase_labels)
        for frame in range(n_frames)
    ]

    print(
        f"{filename}: {n_frames} frames, fps={fps:.2f}, "
        f"phases={phase_labels}, labels={sorted(set(labels))}"
    )

    return labels

In [3]:
def add_labels_to_h5(stamp_df, key_obs_map, h5_dir, pose_type):
    """
    Add labels to h5 files based on the provided time and key observation map.
    Args:
        stamp_df (pd.DataFrame): DataFrame containing the time stamps and behaviors.
        key_obs_map (dict): Mapping of keypoint files to observations.
        h5_dir (str): Directory containing the h5 files.
    """
    key_files = !find {h5_dir} -type f -maxdepth 2
    labeled_files = []
    for pose_file in key_files:
        with h5py.File(pose_file, 'a') as f:
            dataset_name = pose_type+"_labels"
            if dataset_name in f:
                del f[dataset_name]
            if pose_file.endswith(".h5"):
                labels = extract_labels(pose_file, stamp_df, key_obs_map, pose_type)
                if labels is not None:
                    f.create_dataset(dataset_name, data=np.array(labels, dtype='S'))
                    labeled_files.append(pose_file)
    return labeled_files



In [4]:
def create_combined_h5(labeled_files, output_file):
    '''Combine multiple h5 files into a single h5 file with poses and labels.
    Args:
        labeled_files (list): List of paths to h5 files containing poses and labels.
        output_file (str): Path to the output h5 file.
    '''
    pose_list = []
    labels_list = []

    # Load all data first (if fits in memory)
    for fpath in labeled_files:
        with h5py.File(fpath, 'r') as f:
            pose_list.append(f['poses'][:])     # (frames, num_people, keypoints, features)
            labels_list.append(f['labels'][:]) # (frames, label)

    combined_pose = np.concatenate(pose_list, axis=0)
    combined_labels = np.concatenate(labels_list, axis=0)

    # Save combined data to one h5 file
    with h5py.File(output_file, 'a') as f:
        f.create_dataset('poses', data=combined_pose)
        f.create_dataset('labels', data=combined_labels)

In [5]:
all_labeled_files = []

In [6]:
# uwisc

uwisc_stamps = pd.read_csv("/code/jjiang23/pathml/aim2_balance/processed_files/UWisc_timestamps.csv")
uwisc_key_obs_map = pd.read_csv("/code/jjiang23/pathml/aim2_balance/processed_files/UWisc_vid_stamp_map.csv").set_index('Keypoint')['Observation'].to_dict()
#world keypoint
labeled_files = add_labels_to_h5(uwisc_stamps, uwisc_key_obs_map, "/files/pathml/aim2/mediapipe_results/balance_test/heavy/uwisc", pose_type="world_poses")
#camera keypoint
add_labels_to_h5(uwisc_stamps, uwisc_key_obs_map, "/files/pathml/aim2/mediapipe_results/balance_test/heavy/uwisc", pose_type="camera_poses")
all_labeled_files.extend(labeled_files)

RuntimeError: Unable to synchronously get dataspace (identifier is not of specified type)

In [ ]:
#va
va_time = pd.read_csv("/code/jjiang23/pathml/aim2_balance/processed_files/VA_timestamps.csv")
va_key_obs_map = pd.read_csv("/code/jjiang23/pathml/aim2_balance/processed_files/VA_vid_stamp_map.csv").set_index('Keypoint')['Observation'].to_dict()
#world keypoint
labeled_files = add_labels_to_h5(va_time, va_key_obs_map, "/files/pathml/aim2/mediapipe_results/balance_test/heavy/va", pose_type="world_poses")
#camera keypoint
add_labels_to_h5(va_time, va_key_obs_map, "/files/pathml/aim2/mediapipe_results/balance_test/heavy/va", pose_type="camera_poses")
all_labeled_files.extend(labeled_files)


/files/pathml/aim2/mediapipe_results/balance_test/heavy/va/blurred_T18_4SSB.h5 (2394, 2, 33, 5) 29.97 2394 {'Phase1', 'Phase4', 'Phase3', 'Phase2', 'nonphase'}
/files/pathml/aim2/mediapipe_results/balance_test/heavy/va/blurred_T02_4SSBT2.h5 (285, 2, 33, 5) 30.0 285 {'Phase2', 'nonphase'}
/files/pathml/aim2/mediapipe_results/balance_test/heavy/va/blurred_T02_4SSBT1.h5 (700, 2, 33, 5) 30.0 700 {'Phase1', 'nonphase'}
No timestamp observation found for /files/pathml/aim2/mediapipe_results/balance_test/heavy/va/blurred_T02_SSBT3.h5
/files/pathml/aim2/mediapipe_results/balance_test/heavy/va/blurred_T12_4SSB.h5 (3666, 2, 33, 5) 30.0 3666 {'Phase1', 'Phase4', 'Phase3', 'Phase2', 'nonphase'}
/files/pathml/aim2/mediapipe_results/balance_test/heavy/va/blurred_T15_4SSB.h5 (2462, 2, 33, 5) 30.0 2462 {'Phase1', 'Phase4', 'Phase3', 'Phase2', 'nonphase'}
/files/pathml/aim2/mediapipe_results/balance_test/heavy/va/blurred_T11_4SSB.h5 (3843, 2, 33, 5) 30.0 3843 {'Phase1', 'Phase4', 'Phase3', 'Phase2', 'n

In [ ]:
#cp
cp_stamps = pd.read_csv("/code/jjiang23/pathml/aim2_balance/processed_files/CP_timestamps.csv")
cp_key_obs_map = pd.read_csv("/code/jjiang23/pathml/aim2_balance/processed_files/CP_vid_stamp_map.csv").set_index('Keypoint')['Observation'].to_dict()
#world keypoint
labeled_files = add_labels_to_h5(cp_stamps, cp_key_obs_map, "/files/pathml/aim2/mediapipe_results/balance_test/heavy/cp", pose_type="world_poses")
#camera keypoint
add_labels_to_h5(cp_stamps, cp_key_obs_map, "/files/pathml/aim2/mediapipe_results/balance_test/heavy/cp", pose_type="camera_poses")

all_labeled_files.extend(labeled_files)


/files/pathml/aim2/mediapipe_results/balance_test/heavy/cp/P20_Balc.h5 (2008, 2, 33, 5) 29.74390321578797 2008 {'Phase1', 'Phase4', 'Phase3', 'Phase2', 'nonphase'}
/files/pathml/aim2/mediapipe_results/balance_test/heavy/cp/P21_Balc.h5 (2077, 2, 33, 5) 29.743876155493023 2077 {'Phase1', 'Phase4', 'Phase3', 'Phase2', 'nonphase'}
/files/pathml/aim2/mediapipe_results/balance_test/heavy/cp/P22_Balc.h5 (2191, 2, 33, 5) 29.74384454463138 2191 {'Phase1', 'Phase4', 'Phase3', 'Phase2', 'nonphase'}
/files/pathml/aim2/mediapipe_results/balance_test/heavy/cp/P23_Balnc1234.h5 (1983, 2, 33, 5) 29.743884604327338 1983 {'Phase1', 'Phase4', 'Phase3', 'Phase2', 'nonphase'}
/files/pathml/aim2/mediapipe_results/balance_test/heavy/cp/P24_balc.h5 (2435, 2, 33, 5) 29.74378810859231 2435 {'Phase1', 'Phase4', 'Phase3', 'Phase2', 'nonphase'}
/files/pathml/aim2/mediapipe_results/balance_test/heavy/cp/P25_Balc.h5 (2132, 2, 33, 5) 29.743884506672842 2132 {'Phase1', 'Phase4', 'Phase3', 'Phase2', 'nonphase'}
/files/p

In [ ]:
with open("/code/jjiang23/pathml/aim2_balance/processed_files/all_h5_files.txt", "w") as f:
    for file in all_labeled_files:
        f.write(file + "\n")